In [ ]:
CATALOG = "demo_medallion"
BRONZE_SCHEMA = "bronze"
SILVER_SCHEMA = "silver"

spark.sql(f"""
    CREATE CATALOG IF NOT EXISTS {CATALOG}
""")

spark.sql(f"""
    CREATE SCHEMA IF NOT EXISTS {CATALOG}.{BRONZE_SCHEMA}
""")

spark.sql(f"""
    CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SILVER_SCHEMA}
""")

In [ ]:
BRONZE_CUSTOMERS = f"{CATALOG}.{BRONZE_SCHEMA}.customers"

df_bronze = spark.table(BRONZE_CUSTOMERS)

display(df_bronze)

In [ ]:
from pyspark.sql.functions import (
    col,
    trim,
    lower,
    upper
)

df_silver = (
    df_bronze

    # Remove duplicate customer records
    .dropDuplicates(["customer_id"])

    # Normalize text
    .withColumn("customer_name", trim(col("customer_name")))
    .withColumn("email", lower(trim(col("email"))))
    .withColumn("city", trim(col("city")))
    .withColumn("status", upper(trim(col("status"))))

    # Valid customers only
    .filter(col("customer_id").isNotNull())
    .filter(col("email").isNotNull())

    # Valid statuses
    .filter(col("status").isin("ACTIVE", "INACTIVE"))
)

display(df_silver)

In [ ]:
SILVER_CUSTOMERS = f"{CATALOG}.{SILVER_SCHEMA}.customers"

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {SILVER_CUSTOMERS} (
    customer_id INT,
    customer_name STRING,
    email STRING,
    city STRING,
    status STRING
)
USING DELTA
""")

In [ ]:
df_silver.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(SILVER_CUSTOMERS)

print(f"Silver table updated: {SILVER_CUSTOMERS}")